# Citation-graph SNN on the SpikeEngine FPGA

Runs a paper->topic citation classifier (from the `sgnn-superneuro` benchmark) on real
FPGA hardware with on-chip STDP, and cross-checks every paper against a fixed-point-faithful
software reference.

| dataset | neurons | board |
|---|---|---|
| microseer | 90 | Basys3 |
| miniseer | 2116 | ZCU104 |
| cora | 2715 | ZCU104 |
| citeseer | 3318 | ZCU104 |

Set `RUN_HARDWARE = False` to run only the software reference (no board needed).


## 1. Configuration

This notebook needs **two things that are not bundled with the package**:

1. **The dataset dependencies.** The `sgnn-superneuro` checkout brings its own
   third-party requirements (PyYAML, networkx, pandas, wrapt, tqdm,
   scikit-learn, tabulate, matplotlib). Install them in one step:

   ```
   pip install superneuromat[datasets]
   ```

   Without them you will get a clear `missing dependency '<name>'` message
   naming the package to install.

2. **The citation graphs themselves.** These are not redistributed here. Clone
   or download the `sgnn-superneuro` benchmark and point `SGNN_REPO` at the
   checkout containing `gnn_citation_networks.py` and `configs/` — replace the
   `<path-to>` placeholder below.

`PYTHONHASHSEED` is pinned because neuron-to-lane assignment is hash-ordered:
leaving it unset changes the per-lane synapse capacity a network needs from run
to run.

In [ ]:
import os

DATASET      = 'microseer'      # microseer | miniseer | cora | citeseer
BOARD        = None             # None -> dataset's default board, else
                                 # 'basys3' | 'sp701' | 'zcu104'
PORT         = 'auto'
RUN_HARDWARE = False            # True to run on the FPGA
NUM_TEST     = None             # e.g. 20 to limit test papers
os.environ.setdefault('SGNN_REPO', r'<path-to>/sgnn-superneuro-main')
os.environ.setdefault('PYTHONHASHSEED', '0')


## 2. Build the dataset model
Driving weights are rescaled to 2.0 and STDP is global (the hardware's behavior); this is the
validated fixed-point recipe (frac_bits=13, 16-bit weights).


In [ ]:
import sys

from superneuromat.spikeengine.examples import citation_gnn_fpga as cg

repo = cg._find_sgnn_repo(None); sys.path.insert(0, str(repo))
import gnn_citation_networks as G
import xyaml as yaml

board = BOARD or cg.DATASET_BOARD[DATASET]
graph, cfg = cg.build_graph(G, yaml, repo, DATASET, cg.GRAPH_WEIGHT)
papers = graph.selected_papers[:NUM_TEST] if NUM_TEST else graph.selected_papers
print(f'{DATASET}: {len(graph.snn.neuron_thresholds)} neurons, {len(papers)} test papers, board={board}')


## 3. Software reference (fixed-point-faithful)
This is exactly what the hardware is verified against. Run this first to confirm the expected accuracy.


In [ ]:
sw_res = [(cg.software_infer(G, graph, pid), 0) for pid in papers]
sw_acc = G.calculate_accuracy(sw_res, graph.resolution_order, 'SW')
print(f'SOFTWARE one-vs-rest={sw_acc.accuracy:.4f} top1={sw_acc.legacy}/{sw_acc.n}')


## 4. On the FPGA
Connect, then per paper: soft-reset -> load the network -> inject the paper's input -> run with
STDP -> read back the paper->topic weights -> classify. Each paper is cross-checked vs software.


In [ ]:
if RUN_HARDWARE:
    from superneuromat import spikeengine as se
    dev = se.connect(port=PORT, board=board); dev.clear_error()
    hw_res, agree = [], 0
    for i, pid in enumerate(papers):
        hw = cg.hardware_infer(se, dev, graph, pid)
        hw_res.append((hw, 0)); agree += int(sw_res[i][0][1] == hw[1])
    dev.close()
    hw_acc = G.calculate_accuracy(hw_res, graph.resolution_order, 'HW')
    print(f'FPGA one-vs-rest={hw_acc.accuracy:.4f} top1={hw_acc.legacy}/{hw_acc.n}')
    print(f'SW/HW per-paper agreement: {agree}/{len(papers)}')
else:
    print('RUN_HARDWARE is False -- software reference only.')


## 5. Timestep timing (hardware)
Each SuperNeuroMAT tick is a "1 ms" biological timestep; the FPGA computes it in
microseconds. This reads the on-chip cycle counters from the last tick and reports the
wall-clock time per timestep vs the 1 ms budget. (Run section 4 with RUN_HARDWARE=True first.)


In [ ]:
if RUN_HARDWARE:
    from superneuromat import spikeengine as se
    dev2 = se.connect(port=PORT, board=board, dataset=DATASET); dev2.clear_error()
    n = len(graph.snn.neuron_thresholds); pidx = int(graph.paper_neurons[papers[0]].idx)
    dev2.soft_reset()
    info = se.load_network(dev2, graph.snn, frac_bits=cg.FRAC_BITS, weight_w=cg.WEIGHT_W,
                           data_w=cg.DATA_W, stdp_window=cg.STDP_WINDOW, syn_cap_per_lane=cg.get_dataset_cap(DATASET))
    se.run_schedule(dev2, {0:{pidx: cg.INPUT_SPIKE}}, total_ticks=1, frac_bits=cg.FRAC_BITS, n_neurons=n)
    t = cg.measure_timestep(dev2); dev2.close()
    print(f"tick compute : {t['compute_us']:.1f} us ({t['tick_cycles']} cyc)")
    print(f"STDP update  : {t['stdp_us']:.1f} us ({t['stdp_cycles']} cyc)")
    print(f"total/tick   : {t['total_us']:.1f} us = {t['pct_of_1ms']:.1f}% of a 1 ms timestep")
    print('WITHIN 1 ms timestep' if t['within_1ms'] else 'EXCEEDS 1 ms timestep')
else:
    print('set RUN_HARDWARE=True to measure on-chip timestep timing')


---
Equivalent one-liner (any dataset):
```bash
python -m superneuromat.spikeengine.examples.citation_gnn_fpga --dataset miniseer --board zcu104 --port auto
```
